<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/01-why-python.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 1 — Why Python Won

Companion notebook to [Why Python Won](https://www.ai.biz/books/python-primer/why-python/).

The whole argument of the chapter is one measurement: Python is slow, and it does not matter,
because the loop does not run in Python. Let's watch that happen.


## 1. The same computation, two ways


In [ ]:
import time
import numpy as np

n = 1_000_000


In [ ]:
a_list = list(range(n))
b_list = list(range(n))

start = time.perf_counter()
c_list = [a_list[i] + b_list[i] for i in range(n)]
py_time = time.perf_counter() - start
print(f'pure Python: {py_time:.4f}s')


In [ ]:
a = np.arange(n)
b = np.arange(n)

start = time.perf_counter()
c = a + b
np_time = time.perf_counter() - start
print(f'NumPy:       {np_time:.4f}s')
print(f'speedup:     {py_time / np_time:.0f}x')


Nothing about the language changed. What changed is **where the loop lives**.

In the first version the interpreter runs a million times, each iteration checking types,
looking up methods, and allocating boxed integer objects. In the second, Python runs about
once: it hands two memory blocks and an instruction to compiled code.


## 2. Counting your crossings

Every hop from Python into compiled code and back costs overhead. One call over a million
elements is cheap. A million calls over one element each is ruinous.


In [ ]:
data = np.random.default_rng(0).random(200_000)

start = time.perf_counter()
slow = [np.sqrt(x) for x in data]   # 200,000 crossings
t_slow = time.perf_counter() - start

start = time.perf_counter()
fast = np.sqrt(data)                # one crossing
t_fast = time.perf_counter() - start

print(f'element-wise: {t_slow:.4f}s')
print(f'vectorised:   {t_fast:.4f}s')
print(f'speedup:      {t_slow / t_fast:.0f}x')
print('same answer: ', np.allclose(slow, fast))


## 3. Why lists are heavy

A list holds pointers to individual Python objects. An array holds raw values in one block.


In [ ]:
import sys

lst = list(range(1000))
arr = np.arange(1000)

list_bytes = sys.getsizeof(lst) + sum(sys.getsizeof(x) for x in lst)
print(f'list:  {list_bytes:>8,} bytes')
print(f'array: {arr.nbytes:>8,} bytes')
print(f'ratio: {list_bytes / arr.nbytes:.1f}x')


## 4. np.vectorize is not vectorisation

The name promises speed and delivers none. It is a convenience wrapper that loops in Python.


In [ ]:
def scalar_fn(x):
    return x ** 2 + 3 * x + 1

vec = np.vectorize(scalar_fn)

start = time.perf_counter(); vec(data); t_v = time.perf_counter() - start
start = time.perf_counter(); data ** 2 + 3 * data + 1; t_r = time.perf_counter() - start

print(f'np.vectorize:   {t_v:.4f}s')
print(f'real vectorised:{t_r:.4f}s')
print(f'np.vectorize is {t_v / t_r:.0f}x slower')


## Try it yourself

1. Change `n` to 10 million. Does the speedup ratio hold?
2. Compute the dot product of two large vectors both ways. NumPy calls BLAS here, so the gap widens.
3. Write a loop that genuinely cannot be vectorised (each step depends on the last) and time it.
   That is the case where Python is honestly the wrong tool.
